In [ ]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [ ]:
# Load a pretrained transformer model for emotion
# classification. The model predicts the probability
# of seven different emotions for each description.
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None)

In [ ]:
# Define the emotion labels returned by the model.
# These labels will later become new columns
# in the dataset.
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [ ]:
# Predict emotion probabilities for every book
# description and store the results together
# with the corresponding ISBN.
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    # Predict the probability distribution across all supported emotion classes.
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    # Store the predicted emotion scores and the corresponding book identifier.
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [ ]:
# Convert the predicted emotion scores into
# a DataFrame and associate them with each book.
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [ ]:
books = pd.merge(books, emotions_df, on = "isbn13")

In [ ]:
books.to_csv("books_with_emotions.csv", index = False)